# RQ7: Multi-Class Classification — Subscription Tier Prediction

**Research Question:** Can machine learning models accurately classify a customer's subscription tier (Basic, Standard, Premium) from campaign spending behavior and product purchase patterns alone, and what performance metrics best characterise each tier?

**Task:** Multi-Class Classification (3 classes)  
**Target:** `Subscription_Tier` (Basic / Standard / Premium)  
**Models:** Multinomial Logistic Regression, Random Forest, LightGBM  
**Dataset:** Marketing and Product Performance Dataset (Kaggle)

In [103]:
!pip install lightgbm --quiet

In [104]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.preprocessing import label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score, classification_report,
    confusion_matrix, roc_curve, auc, roc_auc_score, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
RANDOM_STATE = 42
OUTPUT_DIR = '/kaggle/working/'

def save_figure(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved figure: {path}')

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'Saved table:  {path}')

print('Imports OK')

Imports OK


## 1. Data Loading & Target Inspection

In [105]:
input_dir = '/kaggle/input'
data_files = [
    os.path.join(root, f)
    for root, dirs, files in os.walk(input_dir)
    for f in files if f.endswith('.xlsx') or f.endswith('.xls') or f.endswith('.csv')
]
print('Found files:', data_files)
FILE_PATH = data_files[0]

df = pd.read_csv(FILE_PATH) if FILE_PATH.endswith('.csv') else pd.read_excel(FILE_PATH)
print(f'Shape: {df.shape}')

df['Has_Flash_Sale'] = df['Flash_Sale_ID'].notna().astype(int)
df['Has_Bundle']     = df['Bundle_ID'].notna().astype(int)
df['Conversion_Rate'] = np.where(df['Clicks'] > 0, df['Conversions'] / df['Clicks'], 0)

TIER_COL = 'Subscription_Tier'
print('Class distribution:')
print(df[TIER_COL].value_counts())

le = LabelEncoder()
df['Tier_Encoded'] = le.fit_transform(df[TIER_COL].fillna('Unknown'))
CLASS_NAMES = list(le.classes_)
N_CLASSES   = len(CLASS_NAMES)
print(f'Classes: {CLASS_NAMES}')

Found files: ['/kaggle/input/datasets/vanishjr/marketing-product-performance/marketing_and_product_performance.csv']
Shape: (10000, 17)
Class distribution:
Subscription_Tier
Basic       3416
Standard    3300
Premium     3284
Name: count, dtype: int64
Classes: ['Basic', 'Premium', 'Standard']


## 2. EDA — Campaign Profiles by Tier

In [106]:
eda_metrics = ['ROI', 'Revenue_Generated', 'Discount_Level', 'Conversion_Rate']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

tiers_sorted = sorted(df[TIER_COL].dropna().unique())
for ax, metric in zip(axes, eda_metrics):
    sns.boxplot(data=df, x=TIER_COL, y=metric, ax=ax,
                palette=PALETTE[:N_CLASSES], order=tiers_sorted)
    ax.set_title(f'{metric} by Subscription Tier', fontsize=11, fontweight='bold')
    ax.set_xlabel('Subscription Tier')

fig.suptitle('RQ7 — Campaign Metrics by Subscription Tier', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq7_metrics_by_tier_boxplots.pdf')
plt.show()

Saved figure: /kaggle/working/rq7_metrics_by_tier_boxplots.pdf


## 3. Preprocessing Pipeline

In [107]:
NUMERIC_FEATURES = [
    'Budget', 'Clicks', 'Conversions', 'Revenue_Generated', 'ROI',
    'Discount_Level', 'Units_Sold', 'Bundle_Price', 'Subscription_Length',
    'Has_Flash_Sale', 'Has_Bundle', 'Conversion_Rate',
    'Customer_Satisfaction_Post_Refund'
]

X = df[NUMERIC_FEATURES].copy()
y = df['Tier_Encoded'].copy()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
preprocessor = ColumnTransformer([('num', numeric_transformer, NUMERIC_FEATURES)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Test class distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}')

Train: (8000, 13) | Test: (2000, 13)
Test class distribution: {np.int64(0): np.int64(683), np.int64(1): np.int64(657), np.int64(2): np.int64(660)}


## 4. Model Training & Cross-Validation

In [108]:
models = {
    'Logistic Regression': LogisticRegression(multi_class='multinomial', solver='lbfgs',
                                               C=1.0, max_iter=1000, class_weight='balanced',
                                               random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=12,
                                                   class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM':            LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                           num_leaves=63, class_weight='balanced',
                                           random_state=RANDOM_STATE, verbosity=-1)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model_results = []
fitted_models = {}

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    t0   = time.time()
    cv_f1 = cross_validate(pipe, X_train, y_train, cv=cv, scoring='f1_macro')
    pipe.fit(X_train, y_train)
    train_time = round(time.time() - t0, 2)
    fitted_models[name] = pipe

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)

    macro_f1  = round(f1_score(y_test, y_pred, average='macro'), 4)
    weight_f1 = round(f1_score(y_test, y_pred, average='weighted'), 4)
    acc       = round(accuracy_score(y_test, y_pred), 4)
    kappa     = round(cohen_kappa_score(y_test, y_pred), 4)
    ovr_auc   = round(roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro'), 4)

    model_results.append({
        'Model': name, 'Accuracy': acc, 'Macro_F1': macro_f1,
        'Weighted_F1': weight_f1, 'Kappa': kappa, 'OVR_AUC': ovr_auc,
        'CV_F1_Mean': round(cv_f1['test_score'].mean(), 4),
        'CV_F1_Std':  round(cv_f1['test_score'].std(), 4),
        'Train_Time_s': train_time
    })
    print(f'{name}: Macro_F1={macro_f1}  AUC={ovr_auc}  Kappa={kappa}')

model_df = pd.DataFrame(model_results)
save_table(model_df, 'rq7_model_comparison.csv')
model_df

Logistic Regression: Macro_F1=0.3267  AUC=0.4873  Kappa=-0.0089
Random Forest: Macro_F1=0.3433  AUC=0.5037  Kappa=0.0148
LightGBM: Macro_F1=0.3345  AUC=0.502  Kappa=0.0015
Saved table:  /kaggle/working/rq7_model_comparison.csv


,Model,Accuracy,Macro_F1,Weighted_F1,Kappa,OVR_AUC,CV_F1_Mean,CV_F1_Std,Train_Time_s
0,Logistic Regression,0.3280,0.3267,0.3270,-0.0089,0.4873,0.3442,0.0080,0.28
1,Random Forest,0.3435,0.3433,0.3434,0.0148,0.5037,0.3342,0.0108,10.92
2,LightGBM,0.3345,0.3345,0.3344,0.0015,0.5020,0.3377,0.0111,13.86


## 5. Per-Class Metrics

In [109]:
best_name = model_df.loc[model_df['Macro_F1'].idxmax(), 'Model']
best_pipe = fitted_models[best_name]
y_pred_best  = best_pipe.predict(X_test)
y_proba_best = best_pipe.predict_proba(X_test)

print(f'Best model: {best_name}')
print(classification_report(y_test, y_pred_best, target_names=CLASS_NAMES))

# Per-class table
Y_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))
per_class_rows = []
for i, cls_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(Y_bin[:, i], y_proba_best[:, i])
    cls_auc = round(auc(fpr, tpr), 4)
    preds_cls = (y_pred_best == i).astype(int)
    true_cls  = (y_test == i).astype(int)
    per_class_rows.append({
        'Class': cls_name,
        'Precision': round(f1_score(true_cls, preds_cls, pos_label=1, average='binary',
                                     zero_division=0), 4),
        'Recall':    round(f1_score(true_cls, preds_cls, pos_label=1, average='binary',
                                     zero_division=0), 4),
        'F1':        round(f1_score(true_cls, preds_cls, pos_label=1, average='binary',
                                     zero_division=0), 4),
        'OVR_AUC':   cls_auc,
        'Support':   int(true_cls.sum())
    })

per_class_df = pd.DataFrame(per_class_rows)
save_table(per_class_df, 'rq7_per_class_metrics.csv')
per_class_df

Best model: Random Forest
              precision    recall  f1-score   support

       Basic       0.34      0.36      0.35       683
     Premium       0.34      0.34      0.34       657
    Standard       0.35      0.33      0.34       660

    accuracy                           0.34      2000
   macro avg       0.34      0.34      0.34      2000
weighted avg       0.34      0.34      0.34      2000

Saved table:  /kaggle/working/rq7_per_class_metrics.csv


,Class,Precision,Recall,F1,OVR_AUC,Support
0,Basic,0.3524,0.3524,0.3524,0.4954,683
1,Premium,0.3397,0.3397,0.3397,0.4998,657
2,Standard,0.3378,0.3378,0.3378,0.5160,660


## 6. Publication-Ready Figures

In [110]:
# ── Normalized Confusion Matrix ──────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_best)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Build annotation with both % and count
annot = np.array([[f'{pct:.1%}\n({cnt})'
                   for pct, cnt in zip(row_pct, row_cnt)]
                  for row_pct, row_cnt in zip(cm_norm, cm)])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5, vmin=0, vmax=1)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'Normalized Confusion Matrix — {best_name} (RQ7)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'rq7_confusion_matrix_normalized.pdf')
plt.show()

Saved figure: /kaggle/working/rq7_confusion_matrix_normalized.pdf


In [111]:
# ── One-vs-Rest ROC Curves ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

fpr_all, tpr_all = {}, {}
for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, PALETTE)):
    fpr, tpr, _ = roc_curve(Y_bin[:, i], y_proba_best[:, i])
    cls_auc     = auc(fpr, tpr)
    fpr_all[i]  = fpr
    tpr_all[i]  = tpr
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{cls_name} (AUC = {cls_auc:.3f})')

# Micro-average
fpr_micro, tpr_micro, _ = roc_curve(Y_bin.ravel(), y_proba_best.ravel())
ax.plot(fpr_micro, tpr_micro, 'k--', linewidth=1.5,
        label=f'Micro-avg (AUC = {auc(fpr_micro, tpr_micro):.3f})')

ax.plot([0, 1], [0, 1], 'grey', linestyle=':', linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'One-vs-Rest ROC Curves — {best_name} (RQ7)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
save_figure(fig, 'rq7_multiclass_roc_curves.pdf')
plt.show()

Saved figure: /kaggle/working/rq7_multiclass_roc_curves.pdf


## 7. Conclusions

In [112]:
best_row = model_df.loc[model_df['Macro_F1'].idxmax()]
print('=' * 60)
print('RQ7 CONCLUSION')
print('=' * 60)
print(f'Best model: {best_row["Model"]}')
print(f'  Macro F1    : {best_row["Macro_F1"]}')
print(f'  OVR AUC     : {best_row["OVR_AUC"]}')
print(f'  Cohen Kappa : {best_row["Kappa"]}')
print()
print('Outputs saved:')
for f in ['rq7_confusion_matrix_normalized.pdf','rq7_multiclass_roc_curves.pdf',
          'rq7_metrics_by_tier_boxplots.pdf',
          'rq7_per_class_metrics.csv','rq7_model_comparison.csv']:
    print(f'  {f}')

RQ7 CONCLUSION
Best model: Random Forest
  Macro F1    : 0.3433
  OVR AUC     : 0.5037
  Cohen Kappa : 0.0148

Outputs saved:
  rq7_confusion_matrix_normalized.pdf
  rq7_multiclass_roc_curves.pdf
  rq7_metrics_by_tier_boxplots.pdf
  rq7_per_class_metrics.csv
  rq7_model_comparison.csv
